<a href="https://colab.research.google.com/github/jbolandDV/responsive-multiples/blob/main/cafb_budget_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Install/Import ---
import pandas as pd
import numpy as np

from google.colab import auth
from google.cloud import bigquery


In [2]:
# --- Authenticate to Google Cloud ---
auth.authenticate_user()

# Set your project (should match where the tables live)
PROJECT_ID = "personalized-pulsing"
client = bigquery.Client(project=PROJECT_ID)

print("Authenticated. Project:", PROJECT_ID)


Authenticated. Project: personalized-pulsing


In [3]:
# --- Parameters (edit as needed) ---
START_DATE = "2024-01-01"
END_DATE   = "2025-12-31"

# Government shutdown window (used only for the 'no shutdown' view)
SHUTDOWN_START = "2025-10-01"
SHUTDOWN_END   = "2025-11-12"

print("Date window:", START_DATE, "to", END_DATE)
print("Shutdown window:", SHUTDOWN_START, "to", SHUTDOWN_END)


Date window: 2024-01-01 to 2025-12-31
Shutdown window: 2025-10-01 to 2025-11-12


In [8]:
BASE_QUERY = f"""
-- CAFB: ALL transactions ({START_DATE} through {END_DATE})
-- If matched to mail: keep mail campaign_type/source_file
-- If NOT matched:
--   - campaign_type = 'digital' when classy_transaction_id is not null
--   - otherwise DROP ROW

WITH t AS (
  SELECT *
  FROM `personalized-pulsing.cafb_reporting.cafb_countable_revenue`
  WHERE DATE(date) BETWEEN DATE('{START_DATE}') AND DATE('{END_DATE}')
),

m AS (
  SELECT *
  FROM `personalized-pulsing.cafb_promotions_clean.mail_files_clean`
  WHERE mail_date IS NOT NULL
    -- mail drops that could possibly match the tx window (10–55 day rule)
    AND mail_date BETWEEN
      DATE_SUB(DATE('{START_DATE}'), INTERVAL 55 DAY)
      AND DATE_SUB(DATE('{END_DATE}'),   INTERVAL 10 DAY)
),

match_candidates AS (
  -- via donor_id
  SELECT
    t.transaction_id,
    m.mail_date,
    m.campaign_name AS source_campaign_name,
    m.source_code   AS source_source_code,
    m.personality,
    m.campaign_type,
    m.client_key,
    m.source_file,

    CASE
      WHEN CAST(t.donor_id AS STRING) = CAST(m.donor_id AS STRING)
           AND t.matchback_map = m.matchback_map THEN 'Both'
      WHEN CAST(t.donor_id AS STRING) = CAST(m.donor_id AS STRING) THEN 'ID Only'
      ELSE 'Error'
    END AS match_method,

    CASE
      WHEN CAST(t.donor_id AS STRING) = CAST(m.donor_id AS STRING)
           AND t.matchback_map = m.matchback_map THEN 1
      WHEN CAST(t.donor_id AS STRING) = CAST(m.donor_id AS STRING) THEN 2
      ELSE 9
    END AS match_rank
  FROM t
  JOIN m
    ON CAST(t.donor_id AS STRING) = CAST(m.donor_id AS STRING)
  WHERE DATE(t.date) >= DATE_ADD(m.mail_date, INTERVAL 10 DAY)
    AND DATE(t.date) <= DATE_ADD(m.mail_date, INTERVAL 55 DAY)

  UNION ALL

  -- via matchback_map
  SELECT
    t.transaction_id,
    m.mail_date,
    m.campaign_name AS source_campaign_name,
    m.source_code   AS source_source_code,
    m.personality,
    m.campaign_type,
    m.client_key,
    m.source_file,

    CASE
      WHEN CAST(t.donor_id AS STRING) = CAST(m.donor_id AS STRING)
           AND t.matchback_map = m.matchback_map THEN 'Both'
      WHEN t.matchback_map = m.matchback_map THEN 'Address Map Only'
      ELSE 'Error'
    END AS match_method,

    CASE
      WHEN CAST(t.donor_id AS STRING) = CAST(m.donor_id AS STRING)
           AND t.matchback_map = m.matchback_map THEN 1
      WHEN t.matchback_map = m.matchback_map THEN 3
      ELSE 9
    END AS match_rank
  FROM t
  JOIN m
    ON t.matchback_map = m.matchback_map
   AND t.matchback_map IS NOT NULL
   AND m.matchback_map IS NOT NULL
   AND t.matchback_map != ''
   AND m.matchback_map != ''
   AND t.matchback_map != '-- (null)'
   AND m.matchback_map != '-- (null)'
  WHERE DATE(t.date) >= DATE_ADD(m.mail_date, INTERVAL 10 DAY)
    AND DATE(t.date) <= DATE_ADD(m.mail_date, INTERVAL 55 DAY)
),

best_match AS (
  SELECT *
  FROM match_candidates
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY transaction_id
    ORDER BY match_rank, mail_date DESC
  ) = 1
)

SELECT
  COALESCE(bm.match_method, 'No Match') AS match_method,

  -- Transaction fields
  t.donor_id,
  t.household_id,
  t.program,
  t.channel,
  t.date AS transaction_date,
  t.amount,
  t.transaction_id,
  t.classy_transaction_id,
  t.type,
  t.payment_method,
  t.postal_code,
  t.matchback_map,
  t.campaign_name,
  t.source_code,
  t.is_recurring,
  t.frequency,
  t.payment_number,

  -- Mail fields (null if no match)
  bm.source_campaign_name,
  bm.source_source_code,
  NULL AS package_name,
  NULL AS package_code,
  bm.personality,

  -- campaign_type (ONLY matched mail type, else digital if classy, else NULL)
  CASE
    WHEN bm.transaction_id IS NOT NULL THEN bm.campaign_type
    WHEN t.classy_transaction_id IS NOT NULL THEN 'digital'
    ELSE NULL
  END AS campaign_type,

  -- acquisition_segment (only for acquisition mail)
  CASE
    WHEN bm.transaction_id IS NOT NULL AND LOWER(bm.campaign_type) = 'acquisition' THEN
      CASE
        WHEN STARTS_WITH(UPPER(COALESCE(bm.source_source_code,'')), 'P') THEN 'prospect'
        WHEN STARTS_WITH(UPPER(COALESCE(bm.source_source_code,'')), 'L') THEN 'lapsed'
        ELSE NULL
      END
    ELSE NULL
  END AS acquisition_segment,

  -- response_type: if mail source_code matches revenue source_code then direct
  CASE
    WHEN bm.transaction_id IS NOT NULL
     AND bm.source_source_code IS NOT NULL
     AND t.source_code = bm.source_source_code
    THEN 'direct'
    ELSE 'matchback'
  END AS response_type,

  -- reply_device
  CASE
    WHEN t.classy_transaction_id IS NOT NULL THEN 'digital'
    ELSE 'analog'
  END AS reply_device,

  -- Source file: keep if matched, else null
  CASE WHEN bm.transaction_id IS NOT NULL THEN bm.source_file ELSE NULL END AS source_file,

  -- Mail timing context (null if no match)
  bm.client_key,
  bm.mail_date AS mail_drop_date,
  DATE_ADD(bm.mail_date, INTERVAL 10 DAY) AS estimated_in_home_date,
  DATE_DIFF(DATE(t.date), DATE_ADD(bm.mail_date, INTERVAL 10 DAY), DAY) AS days_since_in_home

FROM t
LEFT JOIN best_match bm
  USING (transaction_id)

-- DROP non-matched, non-digital rows
WHERE bm.transaction_id IS NOT NULL
   OR t.classy_transaction_id IS NOT NULL
"""
print("Base query length (chars):", len(BASE_QUERY))

Base query length (chars): 5154


In [9]:
def run_to_df(sql: str) -> pd.DataFrame:
    """Run a BigQuery SQL string and return a pandas DataFrame."""
    job = client.query(sql)
    return job.to_dataframe()


In [10]:
df_all = run_to_df(BASE_QUERY)
df_all.head()

,match_method,donor_id,household_id,program,channel,transaction_date,amount,transaction_id,classy_transaction_id,type,...,personality,campaign_type,acquisition_segment,response_type,reply_device,source_file,client_key,mail_drop_date,estimated_in_home_date,days_since_in_home
0,ID Only,669578,0011R00002bX3dvQAC,Direct Response Marketing,Direct Mail,2024-08-22,20.000000000,006Ua00000GkNxaIAF,None,Donation,...,Conscientiousness,appeal,None,direct,analog,CAFB-FY25-July-Post-Card-Mail-File.csv,cafb,2024-07-08,2024-07-18,35
1,Address Map Only,567353,0013600001ymcPtAAI,Direct Response Marketing,Online Donations,2024-07-19,30.000000000,006Ua00000FAdykIAD,None,Donation,...,,appeal,None,matchback,analog,CAFB-FY25-July-Post-Card-Mail-File.csv,cafb,2024-07-08,2024-07-18,1
2,ID Only,471294,0013600000YYvbRAAT,Direct Response Marketing,Online Donations,2025-03-28,25.000000000,006Ua00000QgEagIAF,None,Donation,...,,acquisition,lapsed,matchback,analog,51782_CAFB_0225_Pkg_1.csv,cafb,2025-02-10,2025-02-20,36
3,ID Only,471294,0013600000YYvbRAAT,Direct Response Marketing,Online Donations,2024-09-28,25.000000000,006Ua00000IV4mYIAT,None,Donation,...,,acquisition,lapsed,matchback,analog,50249_CAFB_0924_Pkg_4.csv,cafb,2024-09-09,2024-09-19,9
4,Address Map Only,505771,0013600001W5grXAAR,Direct Response Marketing,Online Donations,2025-02-11,50.000000000,006Ua00000OSTE9IAP,None,Donation,...,,acquisition,lapsed,matchback,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,19


In [11]:
list(df_all)

['match_method',
 'donor_id',
 'household_id',
 'program',
 'channel',
 'transaction_date',
 'amount',
 'transaction_id',
 'classy_transaction_id',
 'type',
 'payment_method',
 'postal_code',
 'matchback_map',
 'campaign_name',
 'source_code',
 'is_recurring',
 'frequency',
 'payment_number',
 'source_campaign_name',
 'source_source_code',
 'package_name',
 'package_code',
 'personality',
 'campaign_type',
 'acquisition_segment',
 'response_type',
 'reply_device',
 'source_file',
 'client_key',
 'mail_drop_date',
 'estimated_in_home_date',
 'days_since_in_home']

# Task
Filter the `df_all` DataFrame to include only rows where `mail_drop_date` is in 2025. Then, calculate the total 'amount' (direct revenue) and count of unique 'transaction_id' (direct gifts) for 'direct' `response_type`, and the total 'amount' (matchback revenue) and count of unique 'transaction_id' (matchback gifts) for 'matchback' `response_type`, both grouped by 'source_campaign_name'. Finally, combine these direct and matchback results into a single table and display it.

## Filter data for mail_drop_date in 2025

### Subtask:
Filter the `df_all` DataFrame to include only rows where `mail_drop_date` falls within the year 2025.


**Reasoning**:
To filter the DataFrame by year, the 'mail_drop_date' column first needs to be converted to datetime objects. After conversion, I will filter the rows where the year is 2025.



**Reasoning**:
The previous code block failed due to a `SyntaxError: unmatched ')'` in the print statement. I will correct the parentheses in the print statement to fix this error.



In [13]:
df_all['mail_drop_date'] = pd.to_datetime(df_all['mail_drop_date'])
df_2025 = df_all[df_all['mail_drop_date'].dt.year == 2025]

print("Shape of original DataFrame:", df_all.shape)
print("Shape of filtered DataFrame (df_2025):", df_2025.shape)
df_2025.head()

Shape of original DataFrame: (113523, 32)
Shape of filtered DataFrame (df_2025): (30573, 32)


,match_method,donor_id,household_id,program,channel,transaction_date,amount,transaction_id,classy_transaction_id,type,...,personality,campaign_type,acquisition_segment,response_type,reply_device,source_file,client_key,mail_drop_date,estimated_in_home_date,days_since_in_home
2,ID Only,471294,0013600000YYvbRAAT,Direct Response Marketing,Online Donations,2025-03-28,25.000000000,006Ua00000QgEagIAF,None,Donation,...,,acquisition,lapsed,matchback,analog,51782_CAFB_0225_Pkg_1.csv,cafb,2025-02-10,2025-02-20,36
4,Address Map Only,505771,0013600001W5grXAAR,Direct Response Marketing,Online Donations,2025-02-11,50.000000000,006Ua00000OSTE9IAP,None,Donation,...,,acquisition,lapsed,matchback,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,19
6,Both,219800,0013600000YZ3CdAAL,Direct Response Marketing,Direct Mail,2025-02-14,25.000000000,006Ua00000OeYefIAF,None,Donation,...,,acquisition,lapsed,direct,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,22
7,Address Map Only,765487,001Ua00000Qz321IAB,Direct Response Marketing,Direct Mail,2025-01-27,200.000000000,006Ua00000NhxNmIAJ,None,Donation,...,,acquisition,prospect,matchback,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,4
10,Address Map Only,766653,001Ua00000S7QYjIAN,Direct Response Marketing,Direct Mail,2025-02-14,20.000000000,006Ua00000Oej3tIAB,None,Donation,...,,acquisition,prospect,direct,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,22


## Calculate direct revenue and gifts

### Subtask:
Calculate the sum of 'amount' (direct_revenue) and count of unique 'transaction_id' (direct_response) for transactions where 'response_type' is 'direct', grouped by 'source_campaign_name'.


**Reasoning**:
I need to filter the DataFrame for 'direct' response types, group by 'source_campaign_name', and then calculate the sum of 'amount' and the count of unique 'transaction_id' to get direct revenue and direct gifts, respectively. Finally, I will rename the columns and store the result.



In [14]:
df_direct_results = df_2025[df_2025['response_type'] == 'direct'].groupby('source_campaign_name').agg(
    direct_revenue=('amount', 'sum'),
    direct_gifts=('transaction_id', 'nunique')
).reset_index()

print("Direct results grouped by source campaign name:")
df_direct_results.head()

Direct results grouped by source campaign name:


,source_campaign_name,direct_revenue,direct_gifts
0,FY25 April Match Appeal,66107.840000000,538
1,FY25 February 2025 Acquisition,7028.000000000,217
2,FY25 Heat or Eat Appeal,55901.220000000,417
3,FY25 January 2025 Acquisition,9813.950000000,286
4,FY25 May 2025 Acquisition,12491.000000000,282


In [15]:
df_matchback_results = df_2025[df_2025['response_type'] == 'matchback'].groupby('source_campaign_name').agg(
    matchback_revenue=('amount', 'sum'),
    matchback_gifts=('transaction_id', 'nunique')
).reset_index()

print("Matchback results grouped by source campaign name:")
df_matchback_results.head()

Matchback results grouped by source campaign name:


,source_campaign_name,matchback_revenue,matchback_gifts
0,FY25 April Match Appeal,108104.690000000,742
1,FY25 February 2025 Acquisition,19606.350000000,243
2,FY25 Heat or Eat Appeal,94773.780000000,630
3,FY25 January 2025 Acquisition,10737.990000000,203
4,FY25 May 2025 Acquisition,34354.520000000,375


In [16]:
df_combined_results = pd.merge(df_direct_results, df_matchback_results, on='source_campaign_name', how='outer').fillna(0)

print("Combined Direct and Matchback Results:")
df_combined_results.head()

Combined Direct and Matchback Results:


,source_campaign_name,direct_revenue,direct_gifts,matchback_revenue,matchback_gifts
0,FY25 April Match Appeal,66107.840000000,538.0,108104.690000000,742
1,FY25 February 2025 Acquisition,7028.000000000,217.0,19606.350000000,243
2,FY25 Heat or Eat Appeal,55901.220000000,417.0,94773.780000000,630
3,FY25 January 2025 Acquisition,9813.950000000,286.0,10737.990000000,203
4,FY25 May 2025 Acquisition,12491.000000000,282.0,34354.520000000,375


In [18]:
print('--- Loading New Donors IDs ---')
NEW_DONORS_QUERY = """
SELECT DISTINCT donor_id
FROM `personalized-pulsing.test.cafb_new_donor_2024`
"""
df_new_donors_raw = run_to_df(NEW_DONORS_QUERY)
new_donors_ids = set(df_new_donors_raw['donor_id'].astype(str).tolist())
print(f"New Donors IDs loaded: {len(new_donors_ids)} unique IDs")

print('\n--- Loading Mode of One IDs ---')
MODE_OF_ONE_QUERY = """
SELECT DISTINCT donor_id
FROM `personalized-pulsing.test.cafb_2024_mode_of_one`
"""
df_mode_of_one_raw = run_to_df(MODE_OF_ONE_QUERY)
mode_of_one_ids = set(df_mode_of_one_raw['donor_id'].astype(str).tolist())
print(f"Mode of One IDs loaded: {len(mode_of_one_ids)} unique IDs")

print('\n--- Loading Responsive Multiples IDs ---')
RESPONSIVE_MULTIPLES_QUERY = """
SELECT DISTINCT donor_id
FROM `personalized-pulsing.test.cafb_2024_responsive_multiples`
"""
df_responsive_multiples_raw = run_to_df(RESPONSIVE_MULTIPLES_QUERY)
responsive_multiples_ids = set(df_responsive_multiples_raw['donor_id'].astype(str).tolist())
print(f"Responsive Multiples IDs loaded: {len(responsive_multiples_ids)} unique IDs")

# Apply donor segments to df_2025
df_2025['donor_id'] = df_2025['donor_id'].astype(str)
df_2025['is_new_donor'] = df_2025['donor_id'].isin(new_donors_ids)
df_2025['is_mode_of_one'] = df_2025['donor_id'].isin(mode_of_one_ids)
df_2025['is_responsive_multiple'] = df_2025['donor_id'].isin(responsive_multiples_ids)

print('\n--- df_2025 with donor segments applied ---')
print(df_2025[['donor_id', 'is_new_donor', 'is_mode_of_one', 'is_responsive_multiple']].head())

--- Loading New Donors IDs ---
New Donors IDs loaded: 5521 unique IDs

--- Loading Mode of One IDs ---
Mode of One IDs loaded: 13693 unique IDs

--- Loading Responsive Multiples IDs ---
Responsive Multiples IDs loaded: 15416 unique IDs

--- df_2025 with donor segments applied ---
   donor_id  is_new_donor  is_mode_of_one  is_responsive_multiple
2    471294         False           False                   False
4    505771         False           False                   False
6    219800         False           False                   False
7    765487         False           False                   False
10   766653         False           False                   False


/tmp/ipython-input-923078477.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025['donor_id'] = df_2025['donor_id'].astype(str)
/tmp/ipython-input-923078477.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025['is_new_donor'] = df_2025['donor_id'].isin(new_donors_ids)
/tmp/ipython-input-923078477.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http

In [19]:
df_all['mail_drop_date'] = pd.to_datetime(df_all['mail_drop_date'])
df_2025 = df_all[df_all['mail_drop_date'].dt.year == 2025].copy()

print("Shape of original DataFrame:", df_all.shape)
print("Shape of filtered DataFrame (df_2025):", df_2025.shape)
df_2025.head()

Shape of original DataFrame: (113523, 32)
Shape of filtered DataFrame (df_2025): (30573, 32)


,match_method,donor_id,household_id,program,channel,transaction_date,amount,transaction_id,classy_transaction_id,type,...,personality,campaign_type,acquisition_segment,response_type,reply_device,source_file,client_key,mail_drop_date,estimated_in_home_date,days_since_in_home
2,ID Only,471294,0013600000YYvbRAAT,Direct Response Marketing,Online Donations,2025-03-28,25.000000000,006Ua00000QgEagIAF,None,Donation,...,,acquisition,lapsed,matchback,analog,51782_CAFB_0225_Pkg_1.csv,cafb,2025-02-10,2025-02-20,36
4,Address Map Only,505771,0013600001W5grXAAR,Direct Response Marketing,Online Donations,2025-02-11,50.000000000,006Ua00000OSTE9IAP,None,Donation,...,,acquisition,lapsed,matchback,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,19
6,Both,219800,0013600000YZ3CdAAL,Direct Response Marketing,Direct Mail,2025-02-14,25.000000000,006Ua00000OeYefIAF,None,Donation,...,,acquisition,lapsed,direct,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,22
7,Address Map Only,765487,001Ua00000Qz321IAB,Direct Response Marketing,Direct Mail,2025-01-27,200.000000000,006Ua00000NhxNmIAJ,None,Donation,...,,acquisition,prospect,matchback,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,4
10,Address Map Only,766653,001Ua00000S7QYjIAN,Direct Response Marketing,Direct Mail,2025-02-14,20.000000000,006Ua00000Oej3tIAB,None,Donation,...,,acquisition,prospect,direct,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,22


**Reasoning**:
Now that `df_2025` is an explicit copy, I will apply the donor segment flags (is_new_donor, is_mode_of_one, is_responsive_multiple) and then create the new `donor_segment` column based on the specified precedence. Finally, I will display the head of the DataFrame including this new column.



In [20]:
df_2025['donor_id'] = df_2025['donor_id'].astype(str)
df_2025['is_new_donor'] = df_2025['donor_id'].isin(new_donors_ids)
df_2025['is_mode_of_one'] = df_2025['donor_id'].isin(mode_of_one_ids)
df_2025['is_responsive_multiple'] = df_2025['donor_id'].isin(responsive_multiples_ids)

def assign_donor_segment(row):
    if row['is_new_donor']:
        return 'new_donor'
    elif row['is_responsive_multiple']:
        return 'responsive_multiples'
    elif row['is_mode_of_one']:
        return 'mode_of_one'
    else:
        return 'other'

df_2025['donor_segment'] = df_2025.apply(assign_donor_segment, axis=1)

print('\n--- df_2025 with donor segments applied and new donor_segment column ---')
print(df_2025[['donor_id', 'is_new_donor', 'is_mode_of_one', 'is_responsive_multiple', 'donor_segment']].head())


--- df_2025 with donor segments applied and new donor_segment column ---
   donor_id  is_new_donor  is_mode_of_one  is_responsive_multiple  \
2    471294         False           False                   False   
4    505771         False           False                   False   
6    219800         False           False                   False   
7    765487         False           False                   False   
10   766653         False           False                   False   

   donor_segment  
2          other  
4          other  
6          other  
7          other  
10         other  


In [21]:
df_2025.head()

,match_method,donor_id,household_id,program,channel,transaction_date,amount,transaction_id,classy_transaction_id,type,...,reply_device,source_file,client_key,mail_drop_date,estimated_in_home_date,days_since_in_home,is_new_donor,is_mode_of_one,is_responsive_multiple,donor_segment
2,ID Only,471294,0013600000YYvbRAAT,Direct Response Marketing,Online Donations,2025-03-28,25.000000000,006Ua00000QgEagIAF,None,Donation,...,analog,51782_CAFB_0225_Pkg_1.csv,cafb,2025-02-10,2025-02-20,36,False,False,False,other
4,Address Map Only,505771,0013600001W5grXAAR,Direct Response Marketing,Online Donations,2025-02-11,50.000000000,006Ua00000OSTE9IAP,None,Donation,...,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,19,False,False,False,other
6,Both,219800,0013600000YZ3CdAAL,Direct Response Marketing,Direct Mail,2025-02-14,25.000000000,006Ua00000OeYefIAF,None,Donation,...,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,22,False,False,False,other
7,Address Map Only,765487,001Ua00000Qz321IAB,Direct Response Marketing,Direct Mail,2025-01-27,200.000000000,006Ua00000NhxNmIAJ,None,Donation,...,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,4,False,False,False,other
10,Address Map Only,766653,001Ua00000S7QYjIAN,Direct Response Marketing,Direct Mail,2025-02-14,20.000000000,006Ua00000Oej3tIAB,None,Donation,...,analog,51680_CAFB_0125_Pkg_1.csv,cafb,2025-01-13,2025-01-23,22,False,False,False,other


# Task
Filter the `df_2025` DataFrame to include only rows where the `campaign_type` is 'appeal'.

## Filter for 'appeal' campaigns

### Subtask:
Filter `df_2025` to include only campaigns where `campaign_type` is 'appeal'.


**Reasoning**:
I need to filter the `df_2025` DataFrame to include only rows where the 'campaign_type' is 'appeal'. Then, I will print the shape of the new DataFrame and display its head to verify the filtering.



In [22]:
df_appeal = df_2025[df_2025['campaign_type'] == 'appeal']

print("Shape of df_appeal:", df_appeal.shape)
df_appeal.head()

Shape of df_appeal: (22922, 36)


,match_method,donor_id,household_id,program,channel,transaction_date,amount,transaction_id,classy_transaction_id,type,...,reply_device,source_file,client_key,mail_drop_date,estimated_in_home_date,days_since_in_home,is_new_donor,is_mode_of_one,is_responsive_multiple,donor_segment
64801,Both,696122,0011R00002n1BLXQA2,Direct Response Marketing,Direct Mail,2025-05-05,100.000000000,006Ua00000SNS8MIAX,None,Donation,...,analog,CAFB-FY25-April-Match-Appeal-Mail-File-Lot-2.csv,cafb,2025-04-21,2025-05-01,4,False,False,True,responsive_multiples
64802,Both,226330,0013600000YZbVnAAL,Direct Response Marketing,Direct Mail,2025-05-22,500.000000000,006Ua00000TBa2fIAD,None,Donation,...,analog,CAFB-FY25-April-Match-Appeal-Mail-File-Lot-2.csv,cafb,2025-04-21,2025-05-01,21,False,False,True,responsive_multiples
64803,Both,198318,0013600000YZX6iAAH,Direct Response Marketing,Direct Mail,2025-05-20,250.000000000,006Ua00000T4tjZIAR,None,Donation,...,analog,CAFB-FY25-April-Match-Appeal-Mail-File-Lot-2.csv,cafb,2025-04-21,2025-05-01,19,False,False,True,responsive_multiples
64804,Both,141393,0013600000YZLlvAAH,Direct Response Marketing,Direct Mail,2025-05-20,30.000000000,006Ua00000T48W9IAJ,None,Donation,...,analog,CAFB-FY25-April-Match-Appeal-Mail-File-Lot-2.csv,cafb,2025-04-21,2025-05-01,19,False,False,True,responsive_multiples
64805,Both,665409,001Ua00000DiIMjIAN,Direct Response Marketing,Direct Mail,2025-05-20,25.000000000,006Ua00000T3tQgIAJ,None,Donation,...,analog,CAFB-FY25-April-Match-Appeal-Mail-File-Lot-2.csv,cafb,2025-04-21,2025-05-01,19,False,False,True,responsive_multiples


## Generate campaign reports

### Subtask:
Iterate through each unique 'appeal' campaign, calculate 'Sent' (if derivable), direct gifts/revenue, matchback gifts/revenue for each donor segment, then merge and calculate total gifts/revenue and response rates. Create a 'Total' row for each campaign and append to a list.


**Reasoning**:
I need to iterate through each unique 'appeal' campaign, calculate direct and matchback gifts/revenue for each donor segment, merge these results, calculate total gifts/revenue and response rates, and then create and append a 'Total' row for each campaign report to a list as specified in the instructions.



**Reasoning**:
The previous code produced a `FutureWarning` related to downcasting during the `fillna` operation. To address this and opt into the future behavior as suggested by pandas, I will set the `future.no_silent_downcasting` option to `True` before the merge and fill operations.



In [32]:
import pandas as pd
import numpy as np

pd.set_option('future.no_silent_downcasting', True)

# Query to get sent quantities from mail_files_clean
SENT_COUNTS_QUERY = f"""
SELECT
    campaign_name AS source_campaign_name,
    COUNT(DISTINCT donor_id) AS sent_quantity
FROM
    `personalized-pulsing.cafb_promotions_clean.mail_files_clean`
WHERE
    mail_date IS NOT NULL
    AND mail_date BETWEEN
      DATE_SUB(DATE('{START_DATE}'), INTERVAL 55 DAY)
      AND DATE_SUB(DATE('{END_DATE}'),   INTERVAL 10 DAY)
GROUP BY
    campaign_name
"""
df_campaign_sent_counts = run_to_df(SENT_COUNTS_QUERY)
print("Campaign sent counts (head):")
print(df_campaign_sent_counts.head())

campaign_reports = []
unique_campaign_names = df_appeal['source_campaign_name'].unique()

for campaign_name in unique_campaign_names:
    current_campaign_df = df_appeal[df_appeal['source_campaign_name'] == campaign_name]

    # Get the overall sent quantity for the current campaign (used for campaign 'Total' row)
    sent_quantity_series_overall = df_campaign_sent_counts[
        df_campaign_sent_counts['source_campaign_name'] == campaign_name
    ]['sent_quantity']
    overall_sent_quantity = sent_quantity_series_overall.iloc[0] if not sent_quantity_series_overall.empty else np.nan

    # Calculate direct revenue and gifts per donor_segment
    direct_results = current_campaign_df[current_campaign_df['response_type'] == 'direct'].groupby('donor_segment').agg(
        direct_revenue=('amount', 'sum'),
        direct_gifts=('transaction_id', 'nunique')
    ).reset_index()

    # Calculate matchback revenue and gifts per donor_segment
    matchback_results = current_campaign_df[current_campaign_df['response_type'] == 'matchback'].groupby('donor_segment').agg(
        matchback_revenue=('amount', 'sum'),
        matchback_gifts=('transaction_id', 'nunique')
    ).reset_index()

    # Merge direct and matchback results
    campaign_segment_report = pd.merge(direct_results, matchback_results, on='donor_segment', how='outer').fillna(0)

    # Now, merge with df_segmented_sends to get segment-specific Sent quantities
    segmented_sends_for_campaign = df_segmented_sends[df_segmented_sends['source_campaign_name'] == campaign_name]
    campaign_segment_report = pd.merge(
        campaign_segment_report,
        segmented_sends_for_campaign[['donor_segment', 'sent_quantity']],
        on='donor_segment',
        how='left' # Use left join to keep all segments from revenue/gifts even if no sends
    ).fillna({'sent_quantity': 0}) # Fill with 0 if a segment had revenue but no recorded sends

    # Rename sent_quantity to 'Sent' for consistency and clarity in the report
    campaign_segment_report = campaign_segment_report.rename(columns={'sent_quantity': 'Sent'})

    # Add source_campaign_name
    campaign_segment_report['source_campaign_name'] = campaign_name

    # Calculate total revenue and gifts
    campaign_segment_report['total_revenue'] = campaign_segment_report['direct_revenue'] + campaign_segment_report['matchback_revenue']
    campaign_segment_report['total_gifts'] = campaign_segment_report['direct_gifts'] + campaign_segment_report['matchback_gifts']

    # Calculate segment-specific response rates using the 'Sent' column from the merge
    # Handle division by zero for response rates, result in NaN if Sent is 0
    campaign_segment_report['direct_response_rate'] = (campaign_segment_report['direct_gifts'] / campaign_segment_report['Sent'].replace(0, np.nan)) * 100
    campaign_segment_report['matchback_response_rate'] = (campaign_segment_report['matchback_gifts'] / campaign_segment_report['Sent'].replace(0, np.nan)) * 100
    campaign_segment_report['total_response_rate'] = (campaign_segment_report['total_gifts'] / campaign_segment_report['Sent'].replace(0, np.nan)) * 100


    # Create a 'Total' row for the current campaign using the overall_sent_quantity
    total_row = {
        'donor_segment': 'Total',
        'source_campaign_name': campaign_name,
        'Sent': overall_sent_quantity, # Use the overall sent quantity for the campaign total
        'direct_revenue': campaign_segment_report['direct_revenue'].sum(),
        'direct_gifts': campaign_segment_report['direct_gifts'].sum(),
        'matchback_revenue': campaign_segment_report['matchback_revenue'].sum(),
        'matchback_gifts': campaign_segment_report['matchback_gifts'].sum(),
        'total_revenue': campaign_segment_report['total_revenue'].sum(),
        'total_gifts': campaign_segment_report['total_gifts'].sum()
    }
    # Calculate response rates for the 'Total' row using the overall_sent_quantity
    total_row['direct_response_rate'] = (total_row['direct_gifts'] / (total_row['Sent'] if total_row['Sent'] > 0 else np.nan)) * 100
    total_row['matchback_response_rate'] = (total_row['matchback_gifts'] / (total_row['Sent'] if total_row['Sent'] > 0 else np.nan)) * 100
    total_row['total_response_rate'] = (total_row['total_gifts'] / (total_row['Sent'] if total_row['Sent'] > 0 else np.nan)) * 100

    campaign_segment_report = pd.concat([campaign_segment_report, pd.DataFrame([total_row])], ignore_index=True)

    campaign_reports.append(campaign_segment_report)

print(f"Generated reports for {len(campaign_reports)} campaigns.")
# Display the first few rows of the first campaign report for verification
if campaign_reports:
    print("\nFirst campaign report (head):")
    print(campaign_reports[0].head(10))

Campaign sent counts (head):
                   source_campaign_name  sent_quantity
0            FY26 December Match Appeal          49425
1              FY25 Meal Voucher Appeal          24050
2  FY25 December Match Follow-Up Appeal          12429
3        FY25 October Newsletter Appeal          29045
4            FY25 July Capital Campaign          34819
Generated reports for 12 campaigns.

First campaign report (head):
          donor_segment   direct_revenue  direct_gifts matchback_revenue  \
0           mode_of_one   1768.000000000            26    5101.940000000   
1             new_donor   4150.000000000            38    7158.620000000   
2                 other   6045.000000000            85   36062.090000000   
3  responsive_multiples  54144.840000000           389   59782.040000000   
4                 Total  66107.840000000           538  108104.690000000   

   matchback_gifts   Sent     source_campaign_name     total_revenue  \
0               39  10945  FY25 April Match A

## Consolidate and format report

### Subtask:
Concatenate all individual campaign reports into a single DataFrame, reorder columns, rename 'donor_segment' to 'Segment', and ensure revenue columns are in float format.


**Reasoning**:
I need to consolidate the individual campaign reports, reorder columns, rename a column, and cast specific columns to float, as per the instructions. First, I'll concatenate the list of campaign reports. Then, I will define the desired column order and reorder the DataFrame's columns. After that, I'll rename 'donor_segment' to 'Segment'. Finally, I'll convert the revenue columns to float.



In [33]:
final_report_df = pd.concat(campaign_reports, ignore_index=True)

desired_column_order = [
    'source_campaign_name', 'Segment', 'Sent', 'direct_revenue', 'direct_gifts',
    'direct_response_rate', 'matchback_revenue', 'matchback_gifts',
    'matchback_response_rate', 'total_revenue', 'total_gifts', 'total_response_rate'
]
final_report_df = final_report_df.rename(columns={'donor_segment': 'Segment'}) # Renaming done here for consistency, but already changed 'donor_segment' to 'Segment' above
final_report_df = final_report_df[desired_column_order]

# Convert numerical columns to appropriate types
revenue_cols = ['direct_revenue', 'matchback_revenue', 'total_revenue']
gifts_cols = ['direct_gifts', 'matchback_gifts', 'total_gifts']
response_rate_cols = ['direct_response_rate', 'matchback_response_rate', 'total_response_rate']

for col in revenue_cols:
    final_report_df[col] = final_report_df[col].astype(float)
for col in gifts_cols:
    final_report_df[col] = final_report_df[col].astype(float)
for col in response_rate_cols:
    final_report_df[col] = final_report_df[col].astype(float)

# Ensure 'Sent' is float as it is used in division
final_report_df['Sent'] = final_report_df['Sent'].astype(float)

print("Consolidated and formatted report (head):")
print(final_report_df.head())
print("\nData types of final report:\n")
print(final_report_df.dtypes)

Consolidated and formatted report (head):
      source_campaign_name               Segment     Sent  direct_revenue  \
0  FY25 April Match Appeal           mode_of_one  10945.0         1768.00   
1  FY25 April Match Appeal             new_donor   5142.0         4150.00   
2  FY25 April Match Appeal                 other  19268.0         6045.00   
3  FY25 April Match Appeal  responsive_multiples   9560.0        54144.84   
4  FY25 April Match Appeal                 Total  44915.0        66107.84   

   direct_gifts  direct_response_rate  matchback_revenue  matchback_gifts  \
0          26.0              0.237551            5101.94             39.0   
1          38.0              0.739012            7158.62             34.0   
2          85.0              0.441146           36062.09            407.0   
3         389.0              4.069038           59782.04            262.0   
4         538.0              1.197818          108104.69            742.0   

   matchback_response_rate  tota

# Task
To extract unique mail drop dates for each source campaign name from `df_appeal` and then generate an Excel report. Each unique campaign will have its own sheet, containing the campaign name and mail drop date as a header, followed by the segmented performance data. The numerical columns will be formatted as currency, integers, or percentages as appropriate.

The report will be saved to `campaign_performance_report_2025.xlsx`.

## Retrieve Mail Dates for Campaigns

### Subtask:
Extract unique mail drop dates for each source campaign name from `df_appeal` to be used in the report headers.


**Reasoning**:
I need to group `df_appeal` by `source_campaign_name`, extract the first unique `mail_drop_date` for each campaign, format it as a string, and store it in a dictionary named `campaign_mail_dates`.



In [35]:
campaign_mail_dates = {}
for campaign_name, group in df_appeal.groupby('source_campaign_name'):
    # Get the first unique mail_drop_date for the campaign
    unique_mail_date = group['mail_drop_date'].dt.strftime('%Y-%m-%d').unique()
    if len(unique_mail_date) > 0:
        campaign_mail_dates[campaign_name] = unique_mail_date[0]

print("Sample of campaign_mail_dates dictionary:")
# Print only a few items to avoid clutter
for i, (key, value) in enumerate(campaign_mail_dates.items()):
    if i >= 5: # Limit to 5 items for display
        break
    print(f"{key}: {value}")


Sample of campaign_mail_dates dictionary:
FY25 April Match Appeal: 2025-04-21
FY25 Heat or Eat Appeal: 2025-02-14
FY25 May Newsletter: 2025-05-19
FY25 Meal Voucher Appeal: 2025-01-17
FY25 Rx or Pantry Appeal: 2025-03-17


**Reasoning**:
Now that the campaign mail drop dates have been extracted and stored, I will proceed to generate the Excel report. This involves initializing an ExcelWriter, iterating through each unique campaign in the `final_report_df`, creating a new sheet for each, writing a header with the campaign name and mail drop date, and then writing the segmented performance data. Finally, I will apply appropriate formatting (currency, integer, percentage) to the numerical columns.



**Reasoning**:
The previous code failed because the `xlsxwriter` module was not found. I need to install `xlsxwriter` using `pip` before it can be used by pandas.



In [39]:
output_file_name = 'campaign_performance_report_2025.xlsx'

# Install xlsxwriter if not already installed
!pip install xlsxwriter

with pd.ExcelWriter(output_file_name, engine='xlsxwriter') as writer:
    workbook = writer.book

    # Define formats
    currency_format = workbook.add_format({'num_format': '$#,##0.00'})
    integer_format = workbook.add_format({'num_format': '#,##0'})
    percentage_format = workbook.add_format({'num_format': '0.00%'}) # Display as percentage with 2 decimal places
    header_format = workbook.add_format({'bold': True, 'font_size': 14})

    # Create a single worksheet for all campaign reports
    worksheet = workbook.add_worksheet('Campaign Performance Summary')

    current_row = 0
    all_columns = None # To collect all unique columns for final width adjustment

    # Sort unique campaign names by mail_drop_date
    sorted_campaign_names = sorted(unique_campaign_names, key=lambda name: campaign_mail_dates.get(name, '9999-12-31'))

    for campaign_name in sorted_campaign_names:
        campaign_data = final_report_df[final_report_df['source_campaign_name'] == campaign_name].copy()

        # Get mail drop date for the header
        mail_date = campaign_mail_dates.get(campaign_name, 'N/A')

        # Write header with campaign name and mail drop date
        header_text = f"Campaign: {campaign_name} (Mail Drop Date: {mail_date})"
        worksheet.write(current_row, 0, header_text, header_format)
        current_row += 2 # Move two rows down for the table headers

        # Exclude 'source_campaign_name' column from the sheet for this table
        campaign_data_to_write = campaign_data.drop(columns=['source_campaign_name'])

        if all_columns is None:
            all_columns = campaign_data_to_write.columns.tolist()

        # Write column headers
        for col_num, value in enumerate(campaign_data_to_write.columns.values):
            worksheet.write(current_row, col_num, value)
        current_row += 1 # Move one row down for the data

        # Write the DataFrame data
        for _, row_data in campaign_data_to_write.iterrows():
            for col_num, col_name in enumerate(campaign_data_to_write.columns):
                cell_value = row_data[col_name]

                # Apply specific formatting
                if col_name in ['direct_revenue', 'matchback_revenue', 'total_revenue']:
                    worksheet.write(current_row, col_num, cell_value, currency_format)
                elif col_name in ['Sent', 'direct_gifts', 'matchback_gifts', 'total_gifts']:
                    worksheet.write(current_row, col_num, int(cell_value) if pd.notna(cell_value) else 0, integer_format)
                elif col_name in ['direct_response_rate', 'matchback_response_rate', 'total_response_rate']:
                    worksheet.write(current_row, col_num, cell_value / 100 if pd.notna(cell_value) else 0, percentage_format)
                else:
                    worksheet.write(current_row, col_num, cell_value)
            current_row += 1 # Move to the next row for the next data entry

        current_row += 2 # Add two blank rows after each campaign report for separation

    # Adjust column widths for better readability across the entire sheet
    if all_columns:
        for i, col_name in enumerate(all_columns):
            # Calculate max length based on column header and data in all rows
            max_len_data = final_report_df[col_name].astype(str).apply(len).max() if col_name in final_report_df.columns else 0
            max_len = max(max_len_data, len(col_name)) + 2
            worksheet.set_column(i, i, max_len)

print(f"Excel report '{output_file_name}' generated successfully.")

Excel report 'campaign_performance_report_2025.xlsx' generated successfully.
